# Qwen3 Embedding 0.6B / 1024 → FAISS HNSW Flat — Kaggle 2 GPU

1. Import notebook này vào Kaggle, bật **Internet** và chọn **GPU T4 x2**.
2. **Add Input** dataset private `nhminh107/sbtc-datav2-etl-20261016` sau khi upload/processing hoàn tất.
3. Chọn **Save Version → Save & Run All** để chạy toàn bộ và lưu Output, rồi tải `.faiss` trong tab Output.

Hai tiến trình embedding chạy song song, mỗi tiến trình giữ một bản model trên một GPU. HNSW Flat được xây **trên CPU** vì FAISS HNSW không có backend GPU. Notebook dừng nếu thiếu 2 GPU, không tự chạy embedding bằng CPU.

Model: [`Qwen/Qwen3-Embedding-0.6B`](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), last-token pooling, 1024 chiều, FP16 khi inference; vector lưu **float32 + L2 normalization**, inner product tương ứng cosine. Model weights và tokenizer được ghim cùng revision; dùng Qwen3 tích hợp sẵn trong Transformers, không tải remote code.

Toàn bộ `children[]` được xử lí, kể cả `retrieval_eligible=false` và text rỗng. Text không có token được thay bằng một EOS token để model tạo embedding thực và tránh chuỗi dài 0; text rỗng vẫn được đánh dấu trong mapping. Chunk quá 8192 token được chia thành các cửa sổ không chồng lấn; lấy trung bình vector theo số token rồi chuẩn hóa lại, giữ một vector/chunk.

Với 4.342.218 chunk: vector float32 chiếm khoảng **16,56 GiB**; HNSW cần khoảng **17,71 GiB** chưa gồm SQLite. Cần khoảng **20–22 GiB RAM trống** cho pha HNSW. Dung lượng đĩa được đo trước embedding: notebook quét corpus để xác minh SHA-256/số lượng, đo chính xác JSONL checkpoint và tạo SQLite thử trên `/tmp` với cùng dữ liệu, thứ tự shard và schema. SQLite thử được dọn sau khi đo; kế hoạch lưu trong `storage_plan.json`. Bước này tốn thêm thời gian đọc/ghi CPU nhưng không tải model hay chạy GPU. Notebook kiểm tra theo số byte đo được cộng **256 MiB dự phòng trên mỗi filesystem**, thay cho mặc định 512 byte/chunk và 1 GiB dự phòng. Vì vậy không còn yêu cầu cố định 21 GiB trên `/kaggle/working`; nếu dữ liệu thực vẫn vượt dung lượng trống, notebook dừng trước embedding và ghi số byte cần thiết trong log.

Checkpoint embedding vẫn nằm trong `/kaggle/working/embedding_checkpoints` và được flush/fsync theo batch để có thể phục hồi như bản mẫu. HNSW/SQLite đang xây nằm trên `/tmp`. Sau khi lưu và xác minh index, notebook kiểm tra kích thước file thực cùng dung lượng sẽ được giải phóng khi chuyển checkpoint sang `/tmp`, rồi mới chuyển checkpoint và đưa kết quả vào Output. Nếu embedding/build lỗi trước bước chuyển, checkpoint vẫn được giữ trong Output. Nếu dùng chung filesystem, cần đủ dung lượng cho các file tồn tại đồng thời. Trạng thái COMPLETE của Kaggle không đồng nghĩa index thành công — xem `pipeline_status.json`, manifest và log.

Nguồn: [model card](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), [FAISS indexes](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes).


In [ ]:
from pathlib import Path
import subprocess
import sys

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Run this notebook on Kaggle with Internet and GPU T4 x2 enabled")

# Install into this Kaggle runtime only; keep the supplied PyTorch/CUDA runtime.
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "faiss-cpu>=1.9,<2", "transformers==4.51.3", "einops>=0.7", "psutil>=5.9",
], check=True)


## Cấu hình

Mặc định chạy toàn bộ corpus đã upload, không giới hạn doc_id hay số chunk. `batch_size` là trần số cửa sổ/GPU; `token_budget` giới hạn số token sau padding mỗi forward. Notebook tự chia nhỏ batch khi CUDA OOM. Có thể tăng `token_budget` nếu VRAM còn dư.

Không cần copy file 23 GB sang `/kaggle/working`. SHA-256 và số lượng document/chunk được xác minh trước embedding. Nếu pha HNSW lỗi, checkpoint vẫn nằm trong Output. Gắn Output phiên trước làm Input hoặc tải checkpoint về, đặt `resume_from` vào thư mục `embedding_checkpoints`, rồi chạy lại. Worker tiếp tục từ batch đã commit; không embedding lại các chunk đã hoàn tất. Không đổi model, chiều hoặc phép chuẩn hóa để giảm RAM.


In [ ]:
import json
import os
import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Select GPU T4 x2 in Kaggle Settings before continuing")
for gpu in range(2):
    print(f"GPU {gpu}: {torch.cuda.get_device_name(gpu)}")

DATASET_SLUG = "sbtc-datav2-etl-20261016"
# Set DATA_PATH manually only if the dataset uses a different mount layout.
DATA_PATH = None
if DATA_PATH is None:
    candidates = [path for path in Path("/kaggle/input").rglob("merged.jsonl")
                  if DATASET_SLUG in str(path)]
    if len(candidates) != 1:
        raise FileNotFoundError(f"Attach {DATASET_SLUG}; expected one merged.jsonl, found {candidates}")
    DATA_PATH = candidates[0]
DATA_PATH = Path(DATA_PATH)
if DATA_PATH.stat().st_size != 23_724_093_649:
    raise ValueError("Dataset size differs from the uploaded ETL corpus")

runner_dir = Path("/tmp/qwen3_hnsw_runner")
runner_dir.mkdir(parents=True, exist_ok=True)
config = {
    "source": str(DATA_PATH),
    "source_sha256": "5b446dab6c1bafb5251441fd18a5ceafcd14837b4e206f4c63c3e80fe9b2adaf",
    "expected_documents": 1_220_598,
    "expected_chunks": 4_342_218,
    "model": "Qwen/Qwen3-Embedding-0.6B",
    "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
    "dimension": 1024,
    "max_length": 8192,
    "batch_size": 64,
    "token_budget": 16_384,
    "buffer_size": 1024,
    "hnsw_m": 32,
    "ef_construction": 80,
    "ef_search": 64,
    "add_batch_size": 10_000,
    "faiss_threads": min(4, os.cpu_count() or 1),
    "scratch": "/kaggle/working/embedding_checkpoints",
    "build_temp": "/tmp/qwen3_hnsw_build",
    "checkpoint_archive": "/tmp/qwen3_hnsw_completed_checkpoints",
    "resume_from": None,  # Optional attached embedding_checkpoints directory.
    "output": "/kaggle/working/faiss_hnsw",
}
config_path = runner_dir / "runner_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
runner_path = runner_dir / "build_qwen3_hnsw.py"
os.environ["HF_HOME"] = "/tmp/huggingface"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Dataset:", DATA_PATH)
print("Output:", config["output"])


## Runner độc lập

Cell này tạo script trong thư mục tạm của Kaggle. Hai worker được khởi chạy bằng process Python riêng để tránh lỗi CUDA/fork và pickle thường gặp khi multiprocessing trực tiếp trong notebook. Không cần tải thêm file Python từ repository.


In [ ]:
runner_source = r'''"""Standalone runner embedded in the Kaggle two-GPU Qwen3/HNSW notebook."""

import argparse
import gc
import hashlib
import json
import os
from pathlib import Path
import shutil
import sqlite3
import subprocess
import sys
import tempfile
from time import monotonic
from itertools import islice

import faiss
import numpy as np
import psutil

CGROUP_ROOT = Path("/sys/fs/cgroup")
DISK_HEADROOM = 256 * 1024**2
MAPPING_SCHEMA = """CREATE TABLE chunks (
            vector_id INTEGER PRIMARY KEY, chunk_id TEXT NOT NULL UNIQUE,
            doc_id INTEGER NOT NULL, parent_id TEXT, chunk_order INTEGER,
            start_char INTEGER, end_char INTEGER, source_line INTEGER,
            empty_text INTEGER NOT NULL, retrieval_eligible INTEGER
        )"""
MAPPING_COLUMNS = (
    "chunk_id", "doc_id", "parent_id", "chunk_order", "start_char",
    "end_char", "source_line", "empty_text", "retrieval_eligible",
)


def write_json_atomic(path: Path, value: dict) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)


def iter_chunks(source: Path, rank: int):
    # Split documents before parsing: each GPU reads half of the JSON objects.
    with source.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if (line_number - 1) % 2 != rank:
                continue
            document = json.loads(line)
            for position, child in enumerate(document.get("children", [])):
                text = child["chunk_text"]
                if not isinstance(text, str):
                    raise ValueError(f"Non-string chunk_text at line {line_number}, child {position}")
                yield {
                    "chunk_id": child["chunk_id"], "doc_id": document["doc_id"],
                    "parent_id": child["parent_id"], "chunk_order": child["chunk_order"],
                    "start_char": child["start_char"], "end_char": child["end_char"],
                    "source_line": line_number, "empty_text": not text.strip(),
                    "retrieval_eligible": child.get("retrieval_eligible"), "text": text,
                }


def padded_batches(order: list[int], lengths: list[int], batch_size: int, token_budget: int):
    offset = 0
    while offset < len(order):
        longest = lengths[order[offset]]
        size = min(batch_size, max(1, token_budget // longest))
        yield order[offset:offset + size]
        offset += size


def normalize_vectors(vectors: np.ndarray) -> np.ndarray:
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if not np.isfinite(vectors).all() or np.any(norms <= 1e-12):
        raise ValueError("Non-finite or zero embedding; no vector was inserted")
    return vectors / norms


def encode_records(records: list[dict], tokenizer, forward, config: dict) -> tuple[np.ndarray, int]:
    tokens = tokenizer(
        [record["text"] for record in records], add_special_tokens=False,
        truncation=False, return_attention_mask=False, return_token_type_ids=False,
    )["input_ids"]
    # Qwen does not add BOS/EOS automatically to document text.
    payload = config["max_length"]
    windows, owners, weights = [], [], []
    long_chunks = 0
    for owner, ids in enumerate(tokens):
        long_chunks += len(ids) > payload
        for start in range(0, max(1, len(ids)), payload):
            piece = ids[start:start + payload]
            # Qwen adds no BOS/CLS automatically; empty inputs need a real token.
            if not piece:
                piece = [tokenizer.eos_token_id]
            windows.append({"input_ids": piece})
            owners.append(owner)
            weights.append(max(1, len(piece)))
    lengths = [len(window["input_ids"]) for window in windows]
    order = sorted(range(len(windows)), key=lengths.__getitem__, reverse=True)
    sums = np.zeros((len(records), config["dimension"]), dtype=np.float32)
    for positions in padded_batches(order, lengths, config["batch_size"], config["token_budget"]):
        vectors = forward([windows[position] for position in positions])
        for position, vector in zip(positions, vectors, strict=True):
            sums[owners[position]] += vector * weights[position]
    return normalize_vectors(sums), long_chunks


def load_encoder(config: dict):
    import torch
    from transformers import AutoModel, AutoTokenizer

    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Each worker must see exactly one usable CUDA GPU")
    torch.set_num_threads(1)
    device = torch.device("cuda:0")
    torch.cuda.synchronize(device)
    tokenizer = AutoTokenizer.from_pretrained(
        config["model"], revision=config["revision"], padding_side="left",
    )
    model = AutoModel.from_pretrained(
        config["model"], revision=config["revision"], attn_implementation="sdpa",
        torch_dtype=torch.float16, low_cpu_mem_usage=False,
    )
    model.config.use_cache = False
    if model.config.hidden_size != config["dimension"]:
        raise ValueError("Unexpected model embedding dimension")
    model.eval().to(device)
    print(f"GPU {config['rank']}: {torch.cuda.get_device_name(0)}; SDPA; float16", flush=True)

    def forward(windows: list[dict]) -> np.ndarray:
        def infer(items: list[dict]) -> np.ndarray:
            features = tokenizer.pad(items, padding=True, return_tensors="pt")
            features = {name: tensor.to(device) for name, tensor in features.items()}
            with torch.inference_mode():
                # Left padding keeps the last real token at the final position.
                embeddings = model(**features, use_cache=False).last_hidden_state[:, -1].float()
            return normalize_vectors(embeddings.cpu().numpy())

        retry = False
        try:
            return infer(windows)
        except torch.cuda.OutOfMemoryError:
            if len(windows) == 1:
                raise RuntimeError("CUDA OOM at batch size 1; no chunk was skipped") from None
            retry = True
        if retry:
            # Release exception frames before retrying smaller batches.
            gc.collect()
            torch.cuda.empty_cache()
            middle = len(windows) // 2
            config["batch_size"] = min(config["batch_size"], middle)
            print(f"GPU {config['rank']}: OOM, retry batch_size={middle}", flush=True)
            return np.concatenate([forward(windows[:middle]), forward(windows[middle:])])

    return tokenizer, forward


def run_worker(config: dict, rank: int) -> None:
    config = {**config, "rank": rank}
    scratch = Path(config["scratch"])
    done = scratch / f"gpu{rank}.done.json"
    if done.exists():
        print(f"GPU {rank}: completed shard already exists", flush=True)
        return
    tokenizer, forward = load_encoder(config)
    smoke = [{"text": text} for text in ("English test.", "Kiểm tra tiếng Việt.", "中文测试。", "", "  ")]
    encode_records(smoke, tokenizer, forward, config)
    progress_path = scratch / f"gpu{rank}.progress.json"
    progress = json.loads(progress_path.read_text()) if progress_path.exists() else {
        "chunks": 0, "long_chunks": 0, "empty_chunks": 0, "vector_bytes": 0, "metadata_bytes": 0,
    }
    count, long_chunks, empty_chunks = (progress[key] for key in ("chunks", "long_chunks", "empty_chunks"))
    resumed_count = count
    if progress["vector_bytes"] != count * config["dimension"] * 4:
        raise ValueError("Checkpoint vector byte count does not match committed chunk count")
    started = monotonic()
    vectors_path, mapping_path = scratch / f"gpu{rank}.f32", scratch / f"gpu{rank}.jsonl"
    for path, key in ((vectors_path, "vector_bytes"), (mapping_path, "metadata_bytes")):
        if progress[key] and (not path.exists() or path.stat().st_size < progress[key]):
            raise ValueError(f"Checkpoint file is shorter than committed progress: {path}")
    with vectors_path.open("r+b" if vectors_path.exists() else "w+b") as vectors_file, (
        mapping_path.open("r+b" if mapping_path.exists() else "w+b")
    ) as metadata_file:
        # Discard only an uncommitted tail left by an interrupted batch.
        for stream, key in ((vectors_file, "vector_bytes"), (metadata_file, "metadata_bytes")):
            stream.seek(progress[key])
            stream.truncate()
        records = []

        def write_batch() -> None:
            nonlocal count, long_chunks, empty_chunks
            vectors, long_count = encode_records(records, tokenizer, forward, config)
            vectors.tofile(vectors_file)
            for record in records:
                record.pop("text")
                empty_chunks += record["empty_text"]
                metadata_file.write((json.dumps(record, ensure_ascii=False) + "\n").encode("utf-8"))
            count += len(records)
            long_chunks += long_count
            for stream in (vectors_file, metadata_file):
                stream.flush()
                os.fsync(stream.fileno())
            write_json_atomic(progress_path, {
                "chunks": count, "long_chunks": long_chunks, "empty_chunks": empty_chunks,
                "vector_bytes": vectors_file.tell(), "metadata_bytes": metadata_file.tell(),
            })
            if count % (config["buffer_size"] * 20) == 0:
                elapsed = monotonic() - started
                print(f"GPU {rank}: {count:,} chunks; {(count - resumed_count) / elapsed:.1f} chunks/s", flush=True)
            records.clear()

        for record in islice(iter_chunks(Path(config["source"]), rank), count, None):
            records.append(record)
            if len(records) == config["buffer_size"]:
                write_batch()
        if records:
            write_batch()
    result = {"chunks": count, "long_chunks": long_chunks, "empty_chunks": empty_chunks,
              "resumed_chunks": resumed_count,
              "seconds": round(monotonic() - started, 2)}
    write_json_atomic(done, result)
    print(f"GPU {rank} complete: {result}", flush=True)


def prepare_source(config: dict) -> dict:
    source = Path(config["source"])
    digest = hashlib.sha256()
    documents = chunks = 0
    with source.open("rb") as stream:
        for line in stream:
            digest.update(line)
            document = json.loads(line)
            documents += 1
            chunks += len(document.get("children", []))
            if documents % 200_000 == 0:
                print(f"Validate: {documents:,} documents; {chunks:,} chunks", flush=True)
    if digest.hexdigest() != config["source_sha256"]:
        raise ValueError("Source SHA-256 differs from the uploaded ETL corpus")
    if documents != config["expected_documents"] or chunks != config["expected_chunks"]:
        raise ValueError(f"Unexpected corpus counts: documents={documents}, chunks={chunks}")
    return {"documents": documents, "chunks": chunks, "sha256": digest.hexdigest()}


def embedding_signature(config: dict) -> dict:
    # HNSW parameters and output paths can change without recomputing embeddings.
    return {key: config[key] for key in (
        "source_sha256", "expected_documents", "expected_chunks", "model", "revision",
        "dimension", "max_length",
    )}


def embed_two_gpus(config: dict, config_path: Path) -> None:
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("Select GPU T4 x2 in Kaggle Settings; two CUDA GPUs are required")
    scratch = Path(config["scratch"])
    scratch.mkdir(parents=True, exist_ok=True)
    signature = scratch / "config.json"
    if signature.exists() and json.loads(signature.read_text()) != embedding_signature(config):
        raise ValueError("Scratch directory belongs to another configuration; choose a new directory")
    write_json_atomic(signature, embedding_signature(config))
    check_disk_space(config)
    counts = prepare_source(config)
    write_json_atomic(scratch / "source.json", counts)
    workers = []
    for rank in range(2):
        environment = {**os.environ, "CUDA_VISIBLE_DEVICES": str(rank),
                       "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
                       "TOKENIZERS_PARALLELISM": "false"}
        workers.append(subprocess.Popen(
            [sys.executable, "-u", __file__, "worker", "--config", str(config_path), "--rank", str(rank)],
            env=environment,
        ))
    try:
        codes = [worker.wait() for worker in workers]
    except BaseException:
        for worker in workers:
            if worker.poll() is None:
                worker.terminate()
        for worker in workers:
            worker.wait()
        raise
    if any(codes):
        raise RuntimeError(f"Embedding workers failed: {codes}; completed shards are kept for retry")


def create_index(dimension: int, count: int, config: dict):
    index = faiss.IndexHNSWFlat(dimension, config["hnsw_m"], faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = config["ef_construction"]
    index.hnsw.efSearch = config["ef_search"]
    # resize then shrink retains C++ capacity, avoiding a second full vector copy.
    storage = faiss.downcast_index(index.storage)
    allocations = [
        (storage.codes, count * dimension * 4),
        (index.hnsw.neighbors, count * (2 * config["hnsw_m"] + 4)),
        (index.hnsw.levels, count), (index.hnsw.offsets, count + 1),
    ]
    for vector, capacity in allocations:
        original_size = vector.size()
        vector.resize(capacity)
        vector.resize(original_size)
    return index


def memory_snapshot() -> dict:
    host = psutil.virtual_memory()
    result = {"host_total": host.total, "host_available": host.available,
              "available": host.available}
    maximum, usage = CGROUP_ROOT / "memory.max", CGROUP_ROOT / "memory.current"
    if maximum.exists() and usage.exists() and maximum.read_text().strip().isdigit():
        limit, current = int(maximum.read_text()), int(usage.read_text())
        statistics = CGROUP_ROOT / "memory.stat"
        stats = dict((key, int(value)) for key, value in (
            line.split() for line in statistics.read_text().splitlines()
        )) if statistics.exists() else {}
        # Disk page cache can be evicted. tmpfs/shmem and dirty pages cannot be
        # treated as immediately reclaimable free RAM.
        file_lru = stats.get("inactive_file", 0) + stats.get("active_file", 0)
        clean_file = max(0, stats.get("file", file_lru) - stats.get("shmem", 0)
                         - stats.get("file_dirty", 0) - stats.get("file_writeback", 0))
        reclaimable = min(file_lru, clean_file)
        available = max(0, min(limit, limit - current + reclaimable))
        result.update(cgroup_limit=limit, cgroup_current=current,
                      reclaimable_file_cache=reclaimable, shmem=stats.get("shmem", 0),
                      available=min(host.available, available))
    return result


def available_memory() -> int:
    return memory_snapshot()["available"]


def index_bytes(config: dict) -> int:
    return config["expected_chunks"] * (4 * config["dimension"] + 4 * (2 * config["hnsw_m"] + 4) + 12)


def check_memory(config: dict) -> dict:
    snapshot = memory_snapshot()
    required = index_bytes(config) + 2 * 1024**3
    print("RAM diagnostics (bytes):", json.dumps({**snapshot, "required": required}), flush=True)
    if snapshot["available"] < required:
        raise RuntimeError(
            f"HNSW needs {required / 1024**3:.2f} GiB including headroom; "
            f"available after reclaimable disk cache: {snapshot['available'] / 1024**3:.2f} GiB"
        )
    return snapshot


def filesystem_type(path: Path) -> str:
    path = path.resolve()
    mounts = [mount for mount in psutil.disk_partitions(all=True)
              if path.is_relative_to(Path(mount.mountpoint))]
    return max(mounts, key=lambda mount: len(mount.mountpoint)).fstype if mounts else "unknown"


def measure_storage(config: dict) -> dict:
    # Measure the exact checkpoint JSONL and final SQLite data before GPU work.
    # Use the same shard order, schema and commit batches as build_index.
    prepare_source(config)
    jsonl_bytes = vector_id = 0
    temporary = Path(config["build_temp"])
    with tempfile.TemporaryDirectory(prefix="storage_probe_", dir=temporary) as directory:
        probe = Path(directory) / "mapping.sqlite"
        with sqlite3.connect(probe) as database:
            database.execute(MAPPING_SCHEMA)
            for rank in range(2):
                rows = []
                for record in iter_chunks(Path(config["source"]), rank):
                    record.pop("text")
                    jsonl_bytes += len((json.dumps(record, ensure_ascii=False) + "\n").encode("utf-8"))
                    rows.append((vector_id, *(record[key] for key in MAPPING_COLUMNS)))
                    vector_id += 1
                    if len(rows) == config["add_batch_size"]:
                        database.executemany("INSERT INTO chunks VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", rows)
                        database.commit()
                        rows.clear()
                    if vector_id % 500_000 == 0:
                        print(f"Storage probe: {vector_id:,}/{config['expected_chunks']:,} mapping rows", flush=True)
                if rows:
                    database.executemany("INSERT INTO chunks VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", rows)
                    database.commit()
            if vector_id != config["expected_chunks"]:
                raise ValueError("Storage probe chunk count differs from source validation")
        sqlite_bytes = probe.stat().st_size
    plan = {
        "signature": embedding_signature(config), "hnsw_m": config["hnsw_m"],
        "jsonl_bytes": jsonl_bytes, "sqlite_bytes": sqlite_bytes,
        "checkpoint_bytes": config["expected_chunks"] * config["dimension"] * 4 + jsonl_bytes,
        "output_bytes": index_bytes(config) + sqlite_bytes,
        "headroom_bytes": DISK_HEADROOM,
    }
    print("Measured storage (bytes):", json.dumps(plan), flush=True)
    return plan


def check_publish_space(config: dict, files: tuple[Path, ...]) -> None:
    output, scratch = Path(config["output"]), Path(config["scratch"])
    destination_device = output.stat().st_dev
    required = sum(path.stat().st_size for path in files if path.stat().st_dev != destination_device)
    reclaimed = 0
    archive = config.get("checkpoint_archive")
    if archive and scratch.stat().st_dev == destination_device:
        if Path(archive).parent.stat().st_dev != destination_device:
            reclaimed = sum(path.stat().st_size for path in scratch.iterdir() if path.is_file())
    free = shutil.disk_usage(output).free
    print(f"Publish budget: files={required / 1024**3:.2f} GiB; "
          f"free after checkpoint move={(free + reclaimed) / 1024**3:.2f} GiB", flush=True)
    if free + reclaimed < required + DISK_HEADROOM:
        raise RuntimeError("Actual index/mapping will not fit in Output; checkpoints have not been moved")


def check_disk_space(config: dict) -> None:
    scratch, output = Path(config["scratch"]), Path(config["output"])
    temporary = Path(config.get("build_temp", config["output"]))
    archive = Path(config.get("checkpoint_archive", config["scratch"])).parent
    plan_path = output / "storage_plan.json"
    if not plan_path.exists():
        raise RuntimeError("Run preflight to measure metadata storage before embedding/build")
    plan = json.loads(plan_path.read_text())
    if plan["signature"] != embedding_signature(config) or plan["hnsw_m"] != config["hnsw_m"]:
        raise ValueError("Storage plan belongs to different source/model/index settings; rerun preflight")
    checkpoint_bytes, output_bytes = plan["checkpoint_bytes"], plan["output_bytes"]
    # During a cross-device move, source and destination coexist until copying
    # finishes. On one device shutil.move renames without another full copy.
    phases = [
        [(scratch, checkpoint_bytes)],
        [(scratch, checkpoint_bytes), (temporary, output_bytes)],
        [(scratch, checkpoint_bytes), (temporary, output_bytes)]
        + ([(archive, checkpoint_bytes)] if scratch.stat().st_dev != archive.stat().st_dev else []),
        [(archive, checkpoint_bytes), (temporary, output_bytes)]
        + ([(output, output_bytes)] if temporary.stat().st_dev != output.stat().st_dev else []),
    ]
    required, paths = {}, {}
    for phase in phases:
        totals = {}
        for path, size in phase:
            device = path.stat().st_dev
            paths[device] = path
            totals[device] = totals.get(device, 0) + size
        for device, size in totals.items():
            required[device] = max(required.get(device, 0), size)
    existing = sum(path.stat().st_size for path in scratch.glob("gpu*.*") if path.is_file())
    for device, size in required.items():
        path = paths[device]
        credit = existing if device == scratch.stat().st_dev else 0
        remaining = max(0, size - credit) + DISK_HEADROOM
        free = shutil.disk_usage(path).free
        print(f"Disk budget: {path}; required free={remaining / 1024**3:.2f} GiB; free={free / 1024**3:.2f} GiB", flush=True)
        if free < remaining:
            raise RuntimeError(f"Not enough disk space on filesystem containing {path}")


def preflight(config: dict) -> None:
    scratch, output = Path(config["scratch"]), Path(config["output"])
    temporary = Path(config["build_temp"])
    archive = Path(config["checkpoint_archive"])
    if archive.resolve().is_relative_to(scratch.resolve()) or scratch.resolve().is_relative_to(archive.resolve()):
        raise ValueError("Checkpoint archive must be separate from active checkpoints")
    if archive.exists():
        raise FileExistsError(f"Checkpoint archive already exists: {archive}")
    if (output / "build_manifest.json").exists():
        raise FileExistsError("A completed index already exists in the output directory")
    for path in (scratch, output, temporary, archive.parent):
        path.mkdir(parents=True, exist_ok=True)
        kind = filesystem_type(path)
        print(f"Storage: {path}; filesystem={kind}; free={shutil.disk_usage(path).free / 1024**3:.1f} GiB", flush=True)
        if kind in {"tmpfs", "ramfs"}:
            raise RuntimeError(f"Use a disk-backed directory instead of {kind}: {path}")
    resume = config.get("resume_from")
    if resume and not (scratch / "config.json").exists():
        previous = Path(resume)
        if json.loads((previous / "config.json").read_text()) != embedding_signature(config):
            raise ValueError("Resume checkpoints use different source/model settings")
        for name in ("source.json", *(f"gpu{rank}.{suffix}" for rank in range(2)
                     for suffix in ("f32", "jsonl", "progress.json", "done.json")), "config.json"):
            if (previous / name).exists():
                shutil.copy2(previous / name, scratch / name)
    snapshot = check_memory(config)
    write_json_atomic(output / "storage_plan.json", measure_storage(config))
    check_disk_space(config)
    # This runs in its own process before loading models. A genuine allocation
    # failure is discovered now, not after several hours of embedding.
    probe = create_index(config["dimension"], config["expected_chunks"], config)
    print(f"Full HNSW capacity allocation passed; RSS={psutil.Process().memory_info().rss / 1024**3:.2f} GiB", flush=True)
    del probe
    gc.collect()
    write_json_atomic(output / "preflight.json", snapshot)
    print("PREFLIGHT PASSED: RAM allocation and disk-backed storage verified", flush=True)


def build_index(config: dict) -> None:
    scratch, output = Path(config["scratch"]), Path(config["output"])
    if json.loads((scratch / "config.json").read_text()) != embedding_signature(config):
        raise ValueError("Embedding configuration does not match the current configuration")
    shards = [json.loads((scratch / f"gpu{rank}.done.json").read_text()) for rank in range(2)]
    count = sum(shard["chunks"] for shard in shards)
    if count != config["expected_chunks"]:
        raise ValueError(f"Embedding count mismatch: {count}")
    check_memory(config)
    output.mkdir(parents=True, exist_ok=True)
    final = output / "qwen3_embedding_0_6b_1024_hnsw_flat.faiss"
    metadata = output / "chunk_mapping.sqlite"
    if any(path.exists() for path in (final, metadata, output / "build_manifest.json")):
        raise FileExistsError("Completed output exists; use a new output directory")
    temporary = Path(config.get("build_temp", config["output"]))
    temporary.mkdir(parents=True, exist_ok=True)
    check_disk_space(config)
    partial_index = temporary / "qwen3_embedding_0_6b_1024_hnsw_flat.faiss.partial"
    partial_metadata = temporary / "chunk_mapping.sqlite.partial"
    if partial_index.exists() or partial_metadata.exists():
        raise FileExistsError("Partial output exists; inspect it or choose a new output directory")
    faiss.omp_set_num_threads(config["faiss_threads"])
    index = create_index(config["dimension"], count, config)
    started = monotonic()
    with sqlite3.connect(partial_metadata) as database:
        database.execute(MAPPING_SCHEMA)
        for rank, shard in enumerate(shards):
            vector_path = scratch / f"gpu{rank}.f32"
            if vector_path.stat().st_size != shard["chunks"] * config["dimension"] * 4:
                raise ValueError(f"Incomplete embedding shard {rank}")
            with vector_path.open("rb") as vectors_file, (
                scratch / f"gpu{rank}.jsonl"
            ).open(encoding="utf-8") as mapping_file:
                processed = 0
                while processed < shard["chunks"]:
                    batch_size = min(config["add_batch_size"], shard["chunks"] - processed)
                    vectors = np.fromfile(vectors_file, dtype=np.float32, count=batch_size * config["dimension"])
                    vectors = vectors.reshape(batch_size, config["dimension"])
                    if not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-4):
                        raise ValueError("Stored embeddings must be finite and L2 normalized")
                    base = index.ntotal
                    rows = []
                    for offset in range(batch_size):
                        record = json.loads(next(mapping_file))
                        rows.append((base + offset, *(record[key] for key in MAPPING_COLUMNS)))
                    database.executemany("INSERT INTO chunks VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", rows)
                    index.add(vectors)
                    database.commit()
                    processed += batch_size
                    elapsed = monotonic() - started
                    print(f"HNSW: {index.ntotal:,}/{count:,}; {index.ntotal / elapsed:.1f} vectors/s", flush=True)
                if mapping_file.readline():
                    raise ValueError(f"Extra mapping rows in shard {rank}")
        if database.execute("SELECT COUNT(*) FROM chunks").fetchone()[0] != index.ntotal:
            raise ValueError("Vector count and metadata count differ")
    faiss.write_index(index, str(partial_index))
    del index
    gc.collect()
    # Reload only after releasing the first index, keeping peak RAM bounded.
    verified = faiss.read_index(str(partial_index))
    if verified.ntotal != count or verified.d != config["dimension"]:
        raise ValueError("Saved FAISS index dimensions or count differ")
    if not isinstance(verified, faiss.IndexHNSWFlat) or verified.metric_type != faiss.METRIC_INNER_PRODUCT:
        raise ValueError("Saved index is not HNSW Flat with inner product")
    del verified
    gc.collect()
    check_publish_space(config, (partial_index, partial_metadata))
    archive = config.get("checkpoint_archive")
    if archive:
        # Keep recovery files in Output on failure. After successful validation,
        # move them outside Output so checkpoint + index do not exceed its quota.
        if Path(archive).exists():
            raise FileExistsError(f"Checkpoint archive already exists: {archive}")
        shutil.move(str(scratch), archive)
    shutil.move(str(partial_metadata), str(metadata))
    shutil.move(str(partial_index), str(final))
    manifest = {**config, "chunks": count, "shards": shards,
                "faiss_version": faiss.__version__, "numpy_version": np.__version__,
                "index_type": "IndexHNSWFlat", "pooling": "last_token", "normalized": True,
                "long_text": "token-window length-weighted mean followed by L2 normalization",
                "vector_order": "GPU 0 shard followed by GPU 1 shard; see SQLite vector_id",
                "build_seconds": round(monotonic() - started, 2), "index_bytes": final.stat().st_size}
    (output / "build_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Ready: {final}; {count:,} vectors", flush=True)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("stage", choices=["preflight", "embed", "worker", "build"])
    parser.add_argument("--config", type=Path, required=True)
    parser.add_argument("--rank", type=int, choices=[0, 1])
    args = parser.parse_args()
    config = json.loads(args.config.read_text(encoding="utf-8"))
    if args.stage == "preflight":
        preflight(config)
    elif args.stage == "embed":
        embed_two_gpus(config, args.config)
    elif args.stage == "worker":
        run_worker(config, args.rank)
    else:
        build_index(config)


if __name__ == "__main__":
    main()
'''
runner_path.write_text(runner_source, encoding="utf-8")
print("Runner written:", runner_path)


## 0. Kiểm tra tài nguyên trước khi embedding
Kiểm tra RAM host/cgroup và filesystem. Xác minh corpus, đo JSONL checkpoint và SQLite bằng metadata thực, không dùng model/GPU. Tính đỉnh dung lượng cho từng filesystem qua các pha embedding, build và chuyển file, cộng 256 MiB dự phòng. Lưu phép đo vào `storage_plan.json`. Sau đó thử cấp phát capacity cho toàn bộ HNSW trong process riêng và thoát để trả RAM. Nếu không đủ tài nguyên, dừng trước khi chạy embedding.


In [ ]:
subprocess.run([
    sys.executable, "-u", str(runner_path), "preflight", "--config", str(config_path),
], check=True)

def run_stage(stage):
    result = subprocess.run([sys.executable, "-u", str(runner_path), stage, "--config", str(config_path)])
    status = {"stage": stage, "return_code": result.returncode, "success": result.returncode == 0}
    (Path(config["output"]) / "pipeline_status.json").write_text(json.dumps(status, indent=2))
    if result.returncode:
        print(f"PIPELINE FAILED at {stage}: return code {result.returncode}. Checkpoints remain in Output.")
    return result.returncode == 0


## 1. Embedding trên hai GPU

Mỗi worker đọc các dòng chẵn/lẻ của nguồn, tokenize theo buffer, xếp cửa sổ theo độ dài để giảm padding, rồi ghi vector và mapping tuần tự ra ổ đĩa. Model được tải một lần/GPU. Log `GPU 0` và `GPU 1` cho biết tốc độ của hai tiến trình. Mỗi batch được commit bằng checkpoint atomic sau khi flush/fsync vector và metadata. Chạy lại sẽ bỏ qua các chunk đã commit và tái sử dụng shard đã hoàn tất với cùng cấu hình.


In [ ]:
embedding_ok = run_stage("embed")


## 2. Xây và lưu HNSW Flat

Pha này chỉ bắt đầu sau khi cả hai GPU hoàn tất và tiến trình model đã thoát, giải phóng RAM/VRAM. FAISS cấp trước capacity cho vector/graph để hạn chế đỉnh RAM khi mở rộng. Đọc từng batch float32 từ scratch, thêm vào index và ghi ánh xạ `vector_id → chunk_id` vào SQLite với ràng buộc ID duy nhất.

Index/SQLite đang xây được ghi ngoài Output trong `/tmp`; vì vậy nếu build thất bại, các file partial lớn không lấn quota của checkpoint. Sau khi lưu, giải phóng index cũ rồi đọc lại `.faiss` để kiểm tra số vector/chiều. Chỉ công bố file cuối khi số mapping khớp `index.ntotal`. Không dùng PQ hay giảm chiều. HNSW là tìm kiếm xấp xỉ, dù vector được lưu đầy đủ.


In [ ]:
build_ok = run_stage("build") if embedding_ok else False


## 3. Tải kết quả

- **`qwen3_embedding_0_6b_1024_hnsw_flat.faiss`**: index cần tải về.
- **`chunk_mapping.sqlite`**: ánh xạ ID trả về từ FAISS sang document/chunk nguồn; cần giữ cùng index.
- **`build_manifest.json`**: model/revision, checksum corpus, tham số và thống kê.

ID của index theo thứ tự shard GPU 0 rồi GPU 1, không phải `doc_id`. Các offsets trong mapping tiếp tục tham chiếu văn bản nguồn trước ETL. Khi tìm kiếm, dùng cùng model/revision, last-token pooling và chuẩn hóa L2 query. Query dùng định dạng `Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: {query}` (\n là ký tự xuống dòng); document không thêm instruction; với query quá dài cần cùng chính sách chia cửa sổ/gộp vector.

Sau **Save Version → Save & Run All**, tải các file từ tab **Output**. Liên kết dưới đây cũng dùng được trong phiên tương tác khi file đã hoàn tất.


In [ ]:
from IPython.display import FileLink, display

output_dir = Path(config["output"])
manifest_path = output_dir / "build_manifest.json"
if build_ok and manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(f"SUCCESS: {manifest['chunks']:,} vectors; index {manifest['index_bytes'] / 1024**3:.2f} GiB")
    for filename in ("qwen3_embedding_0_6b_1024_hnsw_flat.faiss", "chunk_mapping.sqlite", "build_manifest.json"):
        display(FileLink(str(output_dir / filename)))
else:
    print("PIPELINE INCOMPLETE: no completed FAISS index. See pipeline_status.json and runtime logs.")
    print("Recovery checkpoints:", config["scratch"])
    print("Attach/download embedding_checkpoints from Output for the next run; set resume_from to that directory.")
    display(FileLink(str(output_dir / "pipeline_status.json")))
